# L7b: Online SIM Estimation: Updating the Engine as Data Arrive
In this lecture, we let the inputs move. L7a's rebalancing engine recomputed its preferences every day from the market, but the single index model (SIM) parameters it read them from, the intercept and beta of each firm, were frozen at the values estimated on the training decade. Today we ask whether those values were ever stable, replace the batch least-squares fit of L6a with an estimator that updates one observation at a time and forgets at a stated rate, replay the engine on 2025 with frozen and with updated parameters, and then, because one year is one draw, build a scenario engine that runs both versions on many simulated futures and read the distributional scorecard those paths allow. Every step is chronological: no signal or parameter the engine uses on a day includes that day's close (the trade executes at that close, as in L7a), and the one tuning choice we make is made and declared before 2025 is opened.

> __Learning Objectives:__
>
> By the end of this lecture, you should be able to:
> * __Explain why frozen parameters can fail:__ Read a rolling-window beta against its classical pointwise band, say what such bands do and do not establish, and trace what a change in beta or the intercept does to the preference threshold, the basket, and the SIM covariance matrix.
> * __Derive and run the EWLS recursion:__ Write the exponentially weighted least-squares loss, define the decay factor and the half-life, derive the three running moments and their one-step update, solve for the parameters and the residual scale, seed the recursion with a prior of stated weight, and state the units of everything.
> * __Compare frozen and online engines honestly:__ Replay the L7a engine on the realized year without look-ahead, build a SIM scenario generator with the time step in the right place, define the fail rate, lower quantile, tail mean, and drawdown quantile of an ensemble, and say what a scenario ensemble drawn from the frozen model's own parameters can and cannot show.

Let's get started!
___

## Examples
Today, we will be using the following examples to illustrate key concepts:

> [▶ Updating the single index model online with EWLS](CHEME-5660-L7b-Example-EWLS-Replay-Fall-2026.ipynb). Estimate rolling one-year betas with confidence bands for four firms on 2014 to 2024, run the EWLS recursion on close-based growth rates, choose the half-life by a walk-forward comparison of conditional one-step-ahead prediction errors inside the training period and declare it, replay the recursion through 2025 from the archive prior, and read what the update does to the intercepts, betas, preference thresholds, basket, and SIM covariance matrix.

The second example puts both parameter arms into the L7a engine:

> [▶ Frozen versus online parameters on the realized year and on a scenario ensemble](CHEME-5660-L7b-Example-Scenario-Ensemble-Fall-2026.ipynb). Replay 2025 chronologically with frozen and with online SIM parameters and score both against the buy-and-hold baselines, build a SIM scenario generator with the units stated, run both engines on two thousand simulated futures with the same lag and rules, and read the distributional scorecard, the paired differences, and an honest account of what the ensemble shows.

Optional advanced notebooks that extend today's material are listed in the Optional Advanced Material section at the end of this lecture.
___

## Concept Review: The Batch SIM Fit and the Engine That Reads It
In L6a we estimated the SIM of asset $i$, $g_{i,t} = \alpha_{i} + \beta_{i}\,g_{M,t} + \varepsilon_{i,t}$ for trading days $t = 1,\ldots,N$, by least squares on the whole training sample: with the response vector $\mathbf{y} = (g_{i,1},\ldots,g_{i,N})^{\top}$, the design matrix $\hat{\mathbf{X}}\in\mathbb{R}^{N\times 2}$ whose row $t$ is $\mathbf{x}_{t}^{\top} = (1, g_{M,t})$, and the parameter vector $\boldsymbol{\theta}_{i} = (\alpha_{i},\beta_{i})^{\top}$, the estimate solves the normal equations:
$$
\left(\hat{\mathbf{X}}^{\top}\hat{\mathbf{X}}\right)\hat{\boldsymbol{\theta}}_{i} = \hat{\mathbf{X}}^{\top}\mathbf{y}
\qquad\Longleftrightarrow\qquad
\underbrace{\Big(\sum_{t=1}^{N}\mathbf{x}_{t}\mathbf{x}_{t}^{\top}\Big)}_{\text{Gram matrix}}\hat{\boldsymbol{\theta}}_{i} = \underbrace{\sum_{t=1}^{N}\mathbf{x}_{t}\,g_{i,t}}_{\text{cross moments}}
$$
where the second form writes the two matrix products as sums over days, each day entering with the same weight. Every day of 2014 counts as much as every day of 2024, and the answer is one number per parameter for the decade: a single pooled slope over the whole sample. The residual variance $s^{2}_{g,\varepsilon,i} = \lVert\mathbf{r}\rVert_{2}^{2}/(N-2)$ ($\mathbf{r}$ the residual vector) and the classical standard errors follow as in L6a, and the estimates and their standard errors for every security are in the archive that L6b and L7a read.

> __Recall (L7a):__ The engine computes, on each day $t$, the preference weight of every asset from the SIM parameters and two market inputs, the recent market growth $\tilde{g}_{M,t}$ and the market-state signal $\xi_{t}$, both from closes through $t-1$:
> $$
\gamma_{i}(t) = \tanh\left(\frac{\alpha_{i}}{\beta_{i}^{\xi_{t}}} + \beta_{i}^{1-\xi_{t}}\,\tilde{g}_{M,t}\right)
$$
> The sign of $\gamma_{i}(t)$ chooses the basket ($i$ is preferred exactly when $\tilde{g}_{M,t} > -\alpha_{i}/\beta_{i}$), its magnitude the Cobb–Douglas share of the preferred budget, and the engine trades toward those targets at the close of day $t$ on a schedule $\mathcal{R}$ under a turnover cap $\tau_{\max}$ and a drawdown limit $d_{\max}$, paying a cost $c$ per dollar traded from a self-financing account. In L7a the parameters $(\alpha_{i},\beta_{i})$ were the archive values on every day of 2025.

Everything in that box that carries a subscript $t$ moved daily; the two parameters that decide the thresholds did not. Today's question is what happens when they do.
___

## Why Frozen Parameters Can Fail
A pooled estimate is a single slope over its whole sample; if the quantity it summarizes did not sit still, the number was right on no particular day. The way to see whether it sat still is to estimate on windows and to draw the uncertainty of each window's estimate next to it, so that motion can be read against a scale.

> __Rolling estimates with bands:__ Refit the L6a regression on a window of $L$ trading days ending on day $t$ ($L = 252$, one trading year, in the first example), giving $\hat{\beta}_{i}(t)$ and its classical standard error $\text{SE}(\hat{\beta}_{i}(t)) = \sqrt{s^{2}_{g,\varepsilon,i}(t)\left[(\hat{\mathbf{X}}_{t}^{\top}\hat{\mathbf{X}}_{t})^{-1}\right]_{22}}$ from that window's design matrix $\hat{\mathbf{X}}_{t}$ and residual variance $s^{2}_{g,\varepsilon,i}(t)$, and draw the pointwise band $\hat{\beta}_{i}(t)\pm 1.96\,\text{SE}(\hat{\beta}_{i}(t))$. Two labels come attached. It is __descriptive__: the classical formula assumes independent, homoskedastic residuals that daily growth rates violate (L6a), each interval is pointwise, and successive windows overlap in all but one day, so exclusions are correlated. And it gives the motion a __scale__: a rolling estimate wanders inside such bands even when the parameter is constant, and an excursion of many band-widths that lasts for years would be unusual under the classical model, which is what motivates a formal stability test.

On the course data the excursions are of that kind: NVDA's rolling beta ran from about 1.0 to 2.8 across the decade against a full-sample value of 1.75, with median band half-widths of 0.12 to 0.27 across the four firms shown; JNJ's fell from 0.9 toward zero and back; AAPL's moved about 0.4 from its full-sample value in either direction, and GS's ran from 0.8 to 1.8 around a full-sample value of 1.2, for years at a time. This motivates testing a predeclared updating rule; it does not prove drift, and it does not say that updating will improve the engine.

A moving parameter acts on the engine in three places. Write $\mathcal{P}$ for the set of assets ($|\mathcal{P}| = 13$ firms in the examples, as in L5b), $\hat{\boldsymbol{\beta}} = (\hat{\beta}_{1},\ldots,\hat{\beta}_{|\mathcal{P}|})^{\top}$ for the vector of betas, and $\hat{\mathbf{D}}_{g} = \text{diag}(s^{2}_{g,\varepsilon,1},\ldots,s^{2}_{g,\varepsilon,|\mathcal{P}|})$ for the diagonal matrix of residual variances (L6b).

* __The threshold.__ Asset $i$ is preferred when $\tilde{g}_{M,t} > -\alpha_{i}/\beta_{i}$. The frozen thresholds of the thirteen firms sit within a few tenths per year of zero, so the basket hangs on whether the recent market growth is above or below a narrow band; an intercept that moves by a few tenths per year, the size of its sampling error at any memory shorter than a decade (L6a), moves the threshold by more than that band's width and changes which firms are in the basket on a given day.
* __The basket and the weights.__ The magnitude of $\gamma_{i}(t)$, and hence the Cobb–Douglas share, moves with both parameters through the lens $\beta_{i}^{\xi_{t}}$; a beta estimate that goes non-positive breaks the lens (a real power of a non-positive number), which is why L7a stated a rule for that case (assignment to the non-preferred set) and why a fast estimator makes the rule necessary rather than hypothetical.
* __The covariance matrix.__ The L6b input $\hat{\mathbf{\Sigma}}_{g,\text{SIM}} = s^{2}_{g,M}\hat{\boldsymbol{\beta}}\hat{\boldsymbol{\beta}}^{\top} + \hat{\mathbf{D}}_{g}$ carries the betas in the rank-one term and the residual variances on the diagonal. In the first example, holding $s^{2}_{g,M}$ fixed and swapping in the online betas and residual scales moves the matrix, after the identical first day, by 39% to 63% of its Frobenius norm (the square root of the sum of squared entries) on every displayed rebalance day of 2025: an input change that could produce materially different minimum-variance allocations.

None of this says that updating helps; it says that freezing is a modeling choice with consequences, and that an estimator which follows the data needs a stated memory and a stated prior.

> __Example__
>
> [▶ Updating the single index model online with EWLS](CHEME-5660-L7b-Example-EWLS-Replay-Fall-2026.ipynb). Rolling one-year betas of NVDA, AAPL, JNJ, and GS with their bands against the archive and close-based full-sample values; then the EWLS recursion, its half-life, and its 2025 paths.
___

## EWLS: Online SIM Estimation
Exponentially weighted least squares (EWLS) keeps the L6a regression and changes one thing: the weight each day carries. Old observations fade at a stated rate, so the estimate at time $t$ is a least-squares fit to the recent past, and it can be updated one observation at a time without storing the past at all.

> __Weighted loss, decay, and half-life:__ Fix asset $i$. For observations $s = 1,\ldots,t$ let $\mathbf{x}_{s} = (1, g_{M,s})^{\top}$ be the regressor of day $s$ (units of its second entry: inverse years), $y_{s} = g_{i,s}$ the response (inverse years), and $\boldsymbol{\theta} = (\alpha, \beta)^{\top}$ the parameter vector. Choose a __half-life__ $t_{1/2} > 0$ in trading days and set the __decay factor__ $\delta = 2^{-1/t_{1/2}}\in(0,1)$, so that $\delta^{t_{1/2}} = 1/2$: an observation $t_{1/2}$ days old carries half the weight of today's. The EWLS estimate at time $t$ minimizes the exponentially weighted sum of squared residuals:
> $$
L_{t}(\boldsymbol{\theta}) = \sum_{s=1}^{t}\underbrace{\delta^{\,t-s}}_{\text{weight of day }s}\left(y_{s} - \mathbf{x}_{s}^{\top}\boldsymbol{\theta}\right)^{2}
$$
> The total weight of the past, $\sum_{s\le t}\delta^{t-s}\to 1/(1-\delta)\approx t_{1/2}/\ln 2$, is about 91 current-observation equivalents for a one-quarter half-life (a count of weight, not the variance-based effective sample size, which is about twice that). For this data-only loss, ordinary least squares is the case $\delta = 1$: no forgetting.

The loss is quadratic in $\boldsymbol{\theta}$, so its minimizer solves a two-by-two linear system whose coefficients are weighted sums over the past, and those sums update by one multiply and one add per day.

> __Sufficient statistics and the recursion:__ Define the weighted Gram matrix, cross-moment vector, and response second moment:
> $$
\mathbf{A}_{t} = \sum_{s=1}^{t}\delta^{t-s}\,\mathbf{x}_{s}\mathbf{x}_{s}^{\top}\in\mathbb{R}^{2\times 2},\qquad
\mathbf{b}_{t} = \sum_{s=1}^{t}\delta^{t-s}\,\mathbf{x}_{s}\,y_{s}\in\mathbb{R}^{2},\qquad
c_{t} = \sum_{s=1}^{t}\delta^{t-s}\,y_{s}^{2}\in\mathbb{R}
$$
> Setting $\nabla_{\boldsymbol{\theta}}L_{t} = \mathbf{0}$ gives the weighted normal equations $\mathbf{A}_{t}\boldsymbol{\theta} = \mathbf{b}_{t}$. Because every weight of day $s$ at time $t$ is $\delta$ times its weight at time $t-1$, and today's observation enters with weight one, each moment obeys the same one-step update:
> $$
\boxed{
\mathbf{A}_{t} = \delta\,\mathbf{A}_{t-1} + \mathbf{x}_{t}\mathbf{x}_{t}^{\top},\qquad
\mathbf{b}_{t} = \delta\,\mathbf{b}_{t-1} + \mathbf{x}_{t}\,y_{t},\qquad
c_{t} = \delta\,c_{t-1} + y_{t}^{2}\quad\blacksquare}
$$
> The estimate is the solution of the two-by-two system $\mathbf{A}_{t}\hat{\boldsymbol{\theta}}_{i,t} = \mathbf{b}_{t}$ (solved directly; no inverse is formed), unique whenever the weighted market growth rates are not all equal, and the residual scale comes from the same three moments, because at the solved optimum the cross term $\hat{\boldsymbol{\theta}}_{i,t}^{\top}\mathbf{A}_{t}\hat{\boldsymbol{\theta}}_{i,t}$ equals $\hat{\boldsymbol{\theta}}_{i,t}^{\top}\mathbf{b}_{t}$:
> $$
\hat{\sigma}_{\varepsilon,i,t} = \sqrt{\frac{c_{t} - \hat{\boldsymbol{\theta}}_{i,t}^{\top}\mathbf{b}_{t}}{[\mathbf{A}_{t}]_{11}}}
$$
> where $[\mathbf{A}_{t}]_{11} = \sum_{s\le t}\delta^{t-s}$ is the total weight. This is a weighted root mean square residual, not an unbiased residual-variance estimator (no degrees-of-freedom correction; the L6a formula divides by $N-2$); we report it and do not use it for inference.

The three moments $(\mathbf{A}_{t}, \mathbf{b}_{t}, c_{t})$ are __sufficient__: everything the estimator reports at time $t$ is a function of them, the raw history is never stored, and the work per day is constant. The derivation, including the cancellation behind the residual formula, is in the optional advanced notebook.


### The prior
A recursion has to start somewhere, and $\mathbf{A}_{0} = \mathbf{0}$ leaves the first days unidentified. We seed it with the one prior we have, the archive estimate, worth a stated number of pseudo-observations.

> __Prior seeding:__ Given a prior $(\boldsymbol{\theta}_{i,0}, \sigma_{\varepsilon,i,0})$ (the archive's $\hat{\alpha}_{i}$, $\hat{\beta}_{i}$, $s_{g,\varepsilon,i}$), a prior weight $N_{0} > 0$ in pseudo-observations, and the training-period second-moment matrix of the regressor, $\mathbb{E}[\mathbf{x}\mathbf{x}^{\top}] = \begin{pmatrix}1 & g^{\prime}_{M}\\ g^{\prime}_{M} & s^{2}_{g,M} + g^{\prime\,2}_{M}\end{pmatrix}$ built from the archive's training market mean and variance, seed the moments as if $N_{0}$ typical training days consistent with the prior had already been seen:
> $$
\mathbf{A}_{0} = N_{0}\,\mathbb{E}[\mathbf{x}\mathbf{x}^{\top}],\qquad
\mathbf{b}_{0} = \mathbf{A}_{0}\,\boldsymbol{\theta}_{i,0},\qquad
c_{0} = \boldsymbol{\theta}_{i,0}^{\top}\mathbf{b}_{0} + N_{0}\,\sigma_{\varepsilon,i,0}^{2}
$$
> Before any update the recursion returns $\boldsymbol{\theta}_{i,0}$ and $\sigma_{\varepsilon,i,0}$ exactly, and after $t$ updates the prior's weight is $\delta^{t}N_{0}$: it decays like the data. Equivalently, the seeded estimate at time $t$ minimizes the data loss plus a __decaying prior-centered quadratic__, $L_{t}(\boldsymbol{\theta}) + \delta^{t}N_{0}\,(\boldsymbol{\theta}-\boldsymbol{\theta}_{i,0})^{\top}\mathbb{E}[\mathbf{x}\mathbf{x}^{\top}](\boldsymbol{\theta}-\boldsymbol{\theta}_{i,0})$. The data-only loss above and this seeded objective are two objectives; their minimizers approach one another as $\delta^{t}N_{0}$ becomes small against the accumulated data weight, and before that the estimate is a weighted compromise between the archive and the recent past (advanced notebook).

Two remarks keep this honest. First, the ridge estimator of L6a's optional material and this prior both modify the normal equations, but they are different modifications: ridge adds a fixed zero-centered penalty on the slope, the prior shrinks toward the archive with the training geometry $\mathbb{E}[\mathbf{x}\mathbf{x}^{\top}]$ and a penalty that decays, and no equivalence is claimed. Second, every residual receives its age weight, but the information a day carries about the slope scales with the square of its centered market growth (it is what the day adds to the slope direction of $\mathbf{A}_{t}$), and the archive's pseudo-observations carry a market variance estimated on volume-weighted average prices (4.6 per year squared) that are smoother than the closes the recursion updates on (7.5): sixty-three archive pseudo-observations carry roughly the slope information of $63\times 4.6/7.5\approx 39$ average close-based days before any decay, and far fewer in a volatile month, so the prior lets go of the slope faster than $N_{0}$ suggests.


### Units and the algorithm
EWLS consumes __annualized daily growth rates__ from close prices, so nothing is rescaled by the time step $\Delta{t} = 1/252$ years (one trading day) inside the recursion: $\alpha$ and $\hat{\sigma}_{\varepsilon}$ are in inverse years, $\beta$ is dimensionless, $\delta$ is a pure number applied once per observation, $t_{1/2}$ is a count of trading days, $N_{0}$ a count of pseudo-observations, $[\mathbf{A}_{t}]_{11}$ is a weight (dimensionless), $[\mathbf{A}_{t}]_{12} = [\mathbf{A}_{t}]_{21}$ is in inverse years and $[\mathbf{A}_{t}]_{22}$ in inverse years squared, the two entries of $\mathbf{b}_{t}$ are in inverse years and inverse years squared, and $c_{t}$ is in inverse years squared.

__Initialize:__ Given the prior $(\boldsymbol{\theta}_{i,0}, \sigma_{\varepsilon,i,0})$ for each asset $i$, the archive market moments, the weight $N_{0}$, and the half-life $t_{1/2}$ (hence $\delta$), seed $(\mathbf{A}_{0}, \mathbf{b}_{0}, c_{0})$ as above (the same $\mathbf{A}_{0}$ for every asset).

For each new trading day $t$ __do__:
1. After the close, observe the close-based growth rates $g_{M,t}$ and $g_{i,t}$ for every asset; form $\mathbf{x}_{t} = (1, g_{M,t})^{\top}$.
2. Update the moments of every asset: $\mathbf{A}_{t}\leftarrow\delta\mathbf{A}_{t-1} + \mathbf{x}_{t}\mathbf{x}_{t}^{\top}$, $\mathbf{b}_{t}\leftarrow\delta\mathbf{b}_{t-1} + \mathbf{x}_{t}g_{i,t}$, $c_{t}\leftarrow\delta c_{t-1} + g_{i,t}^{2}$.
3. Solve $\mathbf{A}_{t}\hat{\boldsymbol{\theta}}_{i,t} = \mathbf{b}_{t}$ and compute $\hat{\sigma}_{\varepsilon,i,t}$; if the system is numerically unidentified, keep the previous estimate.
4. Hand $(\hat{\alpha}_{i,t}, \hat{\beta}_{i,t}, \hat{\sigma}_{\varepsilon,i,t})$ to the engine for use on day $t+1$, never on day $t$.

__Output:__ The parameter paths $\{(\hat{\alpha}_{i,t}, \hat{\beta}_{i,t}, \hat{\sigma}_{\varepsilon,i,t})\}$ and the final moments. The course package implements the seed, the update, and the whole path (`ewls_init`, `ewls_update!`, `ewls_path`); the first example runs them.
___

## Chronological Replay: Frozen Versus Online
The half-life is the one knob, and it must be chosen without looking at the year we are about to score. We choose it inside the training period by a __walk-forward__ comparison: for each candidate $t_{1/2}\in\{21, 63, 126, 252, \infty\}$, run the recursion over the training growth rates seeded with an OLS fit on the first 252 training growth rates alone (January 6, 2014 to January 5, 2015), and on every day $t$ of 2020 to 2024 predict the firm's growth rate from the parameters estimated through $t-1$ and the market's growth on day $t$, $\hat{g}_{i,t\,|\,t-1} = \hat{\alpha}_{i,t-1} + \hat{\beta}_{i,t-1}\,g_{M,t}$, scoring the mean squared error over the days and the thirteen firms. This is a __conditional__ one-step-ahead evaluation (the parameters are lagged, the day's market growth is not), a test of how well yesterday's parameters describe today's co-movement rather than a tradable forecast. On the course data the one-quarter half-life wins on aggregate (a mean squared error 1.9% below the recursion with no forgetting; one month is too noisy, at 0.5% below; one year sits between) and for seven of the thirteen firms, and we __declare__ $t_{1/2}^{\star} = 63$ trading days before opening 2025. The criterion scores the combined prediction $\hat{\alpha} + \hat{\beta}g_{M,t}$; it neither isolates the intercept nor scores the threshold classification the engine acts on, and nothing downstream re-tunes it.

> __The replay, without look-ahead:__ Both arms run the L7a engine on the 2025 closes with the same rules, schedule, and costs. The __frozen__ arm reads the archive parameters on every day. The __online__ arm restarts the recursion on the first trading day of 2025 from the archive prior with $N_{0} = 63$ and $t_{1/2}^{\star} = 63$, updates it on the close-based growth rates, and on day $t$ reads the parameters that stood before day $t$'s observation was folded in: growth rates through $t-1$, the same one-day shift as the market inputs. On the first day of 2025 the two arms are identical; the parameter paths diverge from the second day, and under the monthly schedule the first day on which the two engines can trade differently is the twenty-second.

What changed in 2025? The online betas moved by half a unit or more within the year and the intercepts by several tenths per year (with a one-quarter memory the intercept is the least identified parameter of the model, as it was in L6a with eleven years); the thresholds moved with them, so the two arms classified at least one firm differently on 101 of 250 days, and the online arm emptied the basket on 12 days against 57 for the frozen arm; one online beta (JNJ, in the April drawdown) went non-positive for three days and was handled by the stated rule. On the year that happened, __updating hurt__: the online monthly engine ended at 1.33 times its initial wealth against 1.53 for the frozen engine, with twice the turnover and one firing of the circuit breaker; on the daily schedule, 1.26 against 1.40. The second example pins the gap to dates and holdings and reruns one counterfactual: in the spring the online intercept of JNJ put that firm alone into a basket the frozen thresholds left empty, and the April fall left the online arm about 58 dollars behind; in November both arms drew down from the same peak day, the frozen engine by 9.0% and the online engine by 10.1%, a tenth of a percent over the limit, so the breaker liquidated the online book, and with the lock covering the December rebalance day it sat in cash through year end; with the breaker disabled the same online engine ends at 1.50, so the firing accounts for 166 of the 202 dollars of the gap. Neither episode is "the rule, not the estimator": the online estimates changed the holdings, the changed holdings crossed the threshold, and the discontinuous rule amplified a tenth of a percent into a quarter of the year in cash. A single path cannot separate the three cleanly, and neither number is a probability, which is why the next section exists.
___

## The Scenario Ensemble
To ask how two policies compare across the years the market __could__ have produced, we need a generator of futures, and the natural one is the model the engine already carries. Calibrated on the training archive, the SIM says how the market moves and how each asset moves with it, and drawing from it gives futures on which both engines can be run under identical rules.

> __SIM scenario generator:__ From a decision date with closes $S_{M}(0)$ for the market and $S_{i}(0)$ for the assets, for days $t = 1,\ldots,N$ (here $N = 250$, the length of 2025) draw the daily market growth rate and the daily residuals independently across days and across assets, all in inverse years:
> $$
g_{M,t}\sim\mathcal{N}\left(g^{\prime}_{M},\,s^{2}_{g,M}\right),\qquad
\varepsilon_{i,t}\sim\mathcal{N}\left(0,\,s^{2}_{g,\varepsilon,i}\right),\qquad
g_{i,t} = \hat{\alpha}_{i} + \hat{\beta}_{i}\,g_{M,t} + \varepsilon_{i,t}
$$
> and compound the closes with the time step $\Delta{t} = 1/252$ years (one trading day): $S_{M}(t) = S_{M}(t-1)\,e^{g_{M,t}\Delta{t}}$ and $S_{i}(t) = S_{i}(t-1)\,e^{g_{i,t}\Delta{t}}$. The market draw is shared by every asset on a day, so the assets co-move through the market term exactly as the SIM covariance says; the residuals are the only other source of variation.

The units deserve one sentence, because they are where such generators go wrong: the SIM is a model of daily growth rates in annualized units, so we draw those, and the time step enters once, in the exponent $g\Delta{t}$ that turns a growth rate into a one-day log return; the market's one-day log return therefore has mean $g^{\prime}_{M}\Delta{t}$ and variance $s^{2}_{g,M}\Delta{t}^{2}$, which is the L4b GBM step with the L5a conversion between the two unit systems. On each simulated future the engines see what they saw on the realized year: the lagged market inputs computed on the joined series (training closes, then the simulated closes) with the same gain and lag, and, for the online arm, the recursion restarted from the archive prior on that future's growth rates with the same shift.

This is a __scenario engine, not a validation__. Its futures have Gaussian daily draws, independent residuals, constant parameters, and moments from volume-weighted average prices that are calmer than close-to-close history; the L3a stylized facts are absent by construction, and nothing in it was checked against how 2025 unfolded. And because the frozen arm's parameters are the generator's own, exactly, the ensemble is a __same-generator__ comparison (L5b): there are no parameter changes for the online estimates to detect, so their movements are responses to sampling noise, and what those movements do to this particular policy's wealth is an empirical question (a nonlinear policy with a breaker is not proved optimal at the true parameters), which the ensemble measures; what it cannot measure is the benefit of updating when the world drifts, because nothing in it drifts. Read with those two sentences attached, the ensemble is exactly what a single year cannot give: a distribution.


### The distributional scorecard
Let $W_{\mathcal{P}}(t)$ be the wealth of an engine's account on day $t$ and $g_{f}$ the risk-free growth rate (L7a). With $n$ simulated futures ($n$ is a count of futures, not the share counts $n_{i}$ of L7a), each of $N$ days ($T = N\Delta{t}$ years), each engine produces $n$ terminal wealths $W^{(1)}_{\mathcal{P}}(T),\ldots,W^{(n)}_{\mathcal{P}}(T)$ and $n$ maximum drawdowns $D^{(1)},\ldots,D^{(n)}$. Sort the terminal wealths as $W_{(1)}\le\cdots\le W_{(n)}$ and fix a tail level, written as a number (5%). The scorecard has these rows:

| Metric | Definition | Reads as |
|:--|:--|:--|
| Median terminal wealth, median NPV | the median of $W^{(p)}_{\mathcal{P}}(T)$; the median of $-W_{\mathcal{P}}(0) + W^{(p)}_{\mathcal{P}}(T)e^{-g_{f}T}$ | the typical outcome, and whether it beat cash at $g_{f}$ |
| Fail rate | the fraction of futures with $W^{(p)}_{\mathcal{P}}(T) < W_{\mathcal{P}}(0)e^{g_{f}T}$, i.e., with negative NPV | how often the policy failed to beat the risk-free baseline |
| Lower quantile at 5% | the order statistic $W_{(k)}$ with $k = \lceil 0.05\,n\rceil$, so that at least 5% of the futures end at or below it | the cutoff below which the worst 5% of the futures ended |
| Tail mean at 5% | the average of the $k$ smallest terminal wealths, $\tfrac{1}{k}\sum_{j=1}^{k}W_{(j)}$ | how bad the worst 5% of the futures were on average |
| Drawdown median and 5% upper cutoff | the median of $D^{(p)}$; the $k$-th largest drawdown (at least 5% of the futures at or above it) | the typical and the bad-case peak-to-trough loss along a path |

The order-statistic convention is stated because with $n = 2000$ and a level of 5% the quantile is the hundredth smallest outcome, a definite number rather than an interpolation, and the tail mean is the average of those hundred. When two policies run on the __same__ futures, as the two arms do, the comparison that matters is __paired__: for each future, the online terminal wealth minus the frozen one, summarized by its median, its spread, and the fraction of futures on which it is positive. Pairing holds the market and residual draws fixed across the two policies (it does not cancel them algebraically, since the policies are nonlinear in the path), which removes most of the path-to-path noise from the comparison and leaves the policy contrast.


### Two thousand futures
Under futures drawn from its own parameters the frozen monthly engine has a median terminal wealth of about 1.14 times initial wealth, fails to beat the risk-free baseline on 27% of the futures, has a 5% lower quantile of 0.93 and a tail mean of 0.89, and a median drawdown of 10.6% that sits at the breaker's limit (a 10% drawdown of a portfolio concentrated in high-beta names is an ordinary event under this generator, and the breaker fires on 63% of the futures); the realized 2025, at 1.53, was a favorable draw for it, not a typical one. The online engine is a little worse on every column (median 1.12, fail rate 31%, lower quantile 0.90, drawdown cutoff 18% against 17%), and SPY bought and held on the same futures is close to both. Paired, the online minus frozen terminal wealth has a median of about $-11$ on a thousand with a standard deviation of about 140, and the online arm ends ahead on 44% of the futures, materially below half: the observed cost of updating under a generator with no drift for the estimator to learn.

> __Example__
>
> [▶ Frozen versus online parameters on the realized year and on a scenario ensemble](CHEME-5660-L7b-Example-Scenario-Ensemble-Fall-2026.ipynb). The realized-path scorecard for both arms on both schedules with the GMV, tangent, equal-weight, and SPY baselines run through the same engine (frozen monthly 1.53, online monthly 1.33, GMV 1.38, tangent 1.28, equal weights 1.59, SPY 1.17), the event table and the breaker-off counterfactual, the generator and its checks, two thousand futures of 250 days, the distributional scorecard for the frozen engine, the online engine, and SPY, and the paired-difference histogram.
___

## Validation Gates: A Pointer
The comparisons of this lecture, frozen against online on a realized path and on an ensemble, become in L15a a set of __deployment gates__: pass-or-fail rules with stated limits (a median terminal wealth that must beat the frozen baseline, a fail rate and a tail that must not exceed set values, a drawdown and a turnover budget), evaluated on the same paired ensemble and reported before an online engine is allowed to replace its frozen ancestor. Nothing here is such a gate; L15a turns these comparisons into pass or fail.
___

## Optional Advanced Material
The notebook below extends today's material. It is optional and is not a prerequisite for L8b; the [advanced index](advanced/README.md) describes it.

* [▶ Derivation: the EWLS recursion and the decaying prior](advanced/online_learning/CHEME-5660-L7b-Advanced-EWLS-Recursion-Fall-2026.ipynb). Derive the weighted normal equations and the three sufficient statistics from the exponentially weighted loss, show the one-step recursion, solve for the parameters and the residual scale (with the cancellation that makes the residual formula work), and prove that the prior seeding returns the prior exactly and that the seeded recursion minimizes the data loss plus a decaying prior-centered quadratic.
___

## Summary
In this lecture, we showed that the archive's pooled SIM parameters can mask time variation the rolling estimates make visible, replaced the batch fit with the EWLS recursion (a decay factor, a half-life, three running moments, one prior of stated weight), replayed the L7a engine on 2025 with frozen and with online parameters under a half-life declared inside the training period, and built a SIM scenario generator whose futures gave both engines a distributional scorecard and a paired comparison.

> __Key Takeaways:__
>
> * __A frozen parameter is a modeling choice, and rolling bands give the motion a scale:__ Rolling estimates with classical pointwise bands show the course betas several band-widths from their pooled values for years at a time, which motivates testing an updating rule without proving drift; a moving intercept or beta moves the preference threshold, the basket, and the SIM covariance matrix, and a fast estimator makes the non-positive-beta rule necessary rather than hypothetical.
> * __EWLS is least squares with a memory, and everything about it is stated:__ The decay factor sets the half-life, three running moments are sufficient and update in one step per day, the parameters come from a two-by-two solve and the residual scale from the same moments, the prior enters as a stated number of pseudo-observations that decay like the data, the units are annualized growth rates, and the one tuning choice, the half-life, is made by a walk-forward comparison inside the training period and declared before the test year.
> * __Compare policies on many futures, and say what the futures are:__ On the realized 2025 the online engine finished behind the frozen one, mostly through one breaker firing that a counterfactual measures and that a single path cannot cleanly attribute; on two thousand futures drawn from the SIM the frozen engine's typical outcome was far below its 2025 result and the online engine paid a small price for updating under a generator with no drift to learn; the ensemble is a scenario engine calibrated on the training decade, the comparison is paired, its generator is the frozen model's own parameters, and none of it is validation.

Next time, in L8b, we leave the portfolio process for derivatives and begin with the option contracts themselves; the frozen-versus-online comparison returns in L15a as a set of pass-or-fail deployment gates.
___

## Disclaimer and Risks
__This content is offered solely for training and informational purposes__. No offer or solicitation to buy or sell securities or derivative products or any investment or trading advice or strategy is made, given, or endorsed by the teaching team. 

__Trading involves risk__. Carefully review your financial situation before investing in securities, futures contracts, options, or commodity interests. Past performance, whether actual or indicated by historical tests of strategies, is no guarantee of future performance or success. Trading is generally inappropriate for someone with limited resources, investment or trading experience, or a low-risk tolerance. Only risk capital that is not required for living expenses should be used.

__You are fully responsible for any investment or trading decisions you make__. Such decisions should be based solely on evaluating your financial circumstances, investment or trading objectives, risk tolerance, and liquidity needs.

___